In [1]:
import numpy as np

In [2]:
import torch

print(torch.__version__)

2.10.0+cu130


In [4]:
X = torch.tensor([[3, 2, 1], [5, 6, 7]])
X

tensor([[3, 2, 1],
        [5, 6, 7]])

In [5]:
print(X.shape)
X.dtype

torch.Size([2, 3])


torch.int64

In [6]:
X[:, 2]

tensor([1, 7])

In [7]:
print( 10 * (X+1))
print(X.mean(dtype=torch.float32))
print(X.exp())
print(X.max(dim=0)) #axis=0

tensor([[40, 30, 20],
        [60, 70, 80]])
tensor(4.)
tensor([[  20.0855,    7.3891,    2.7183],
        [ 148.4132,  403.4288, 1096.6332]])
torch.return_types.max(
values=tensor([5, 6, 7]),
indices=tensor([1, 1, 1]))


In [8]:
torch.FloatTensor(np.array([[1, 2, 3], [3, 2, 1]]))

tensor([[1., 2., 3.],
        [3., 2., 1.]])

In [9]:
X[:, 1] = -99
print(X)
X.relu_()
X

tensor([[  3, -99,   1],
        [  5, -99,   7]])


tensor([[3, 0, 1],
        [5, 0, 7]])

In [10]:
device = 'empty'
if torch.cuda.is_available():
    device = 'cuda'
    
device

'cuda'

In [11]:
M = torch.rand((1000, 1000), dtype=torch.float32)
%timeit M @ M.T

M = torch.tensor((1000, 1000), device='cuda', dtype=torch.float32)
%timeit M @ M.T

2.59 ms ± 195 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


<magic-timeit>:1: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4480.)


8.65 μs ± 160 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [12]:
C = torch.rand((10, 10))
C = C.to('cuda')
C.device

device(type='cuda', index=0)

#### autograd

In [13]:
x = torch.tensor(5.0, requires_grad=True)
f = x**2
print(f)

f.backward()
x.grad

tensor(25., grad_fn=<PowBackward0>)


tensor(10.)

In [14]:
lr = 0.1
with torch.no_grad():
    x -= lr * x.grad
    
x

tensor(4., requires_grad=True)

In [15]:
x_detached = x.detach()
x_detached -= lr * x.grad

x_detached 

tensor(3.)

In [16]:
x.grad.zero_()

tensor(0.)

In [17]:
lr = 0.1
x = torch.tensor(5.0, dtype=torch.float32, requires_grad=True)

for i in range(100):
    f = x**2
    f.backward()
    with torch.no_grad():
        x -= lr * x.grad
        
    x.grad.zero_()
    
x

tensor(1.0185e-09, requires_grad=True)

In [4]:
#california datasets
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(housing.data, housing.target, test_size=0.2, random_state=52)

X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=52)

print(X_train.shape)
y_train.shape

(13209, 8)


(13209,)

In [5]:
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
X_valid = torch.FloatTensor(X_valid)
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)

means = X_train.mean(dim=0, keepdim=True)
stds = X_train.std(dim=0, keepdim=True)

X_train = (X_train - means) / stds
X_test = (X_test - means) / stds
X_valid = (X_valid - means) / stds

In [6]:
#linear regression model
torch.manual_seed(52)
n_features = X_train.shape[1]

w = torch.randn((n_features, 1), dtype=torch.float32, requires_grad=True)
b = torch.tensor(0., dtype=torch.float32, requires_grad=True)

In [7]:
lr = 0.3
for epoch in range(30):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train)**2).mean()
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    print("epocha #",epoch, "Loss:", loss.item())

epocha # 0 Loss: 12.10840892791748
epocha # 1 Loss: 2.1366236209869385
epocha # 2 Loss: 0.9621390104293823
epocha # 3 Loss: 0.728757381439209
epocha # 4 Loss: 0.6595050692558289
epocha # 5 Loss: 0.630736231803894
epocha # 6 Loss: 0.6146693825721741
epocha # 7 Loss: 0.603450357913971
epocha # 8 Loss: 0.5944903492927551
epocha # 9 Loss: 0.5868364572525024
epocha # 10 Loss: 0.5800967812538147
epocha # 11 Loss: 0.5740844011306763
epocha # 12 Loss: 0.5686913728713989
epocha # 13 Loss: 0.5638428330421448
epocha # 14 Loss: 0.5594794154167175
epocha # 15 Loss: 0.555550754070282
epocha # 16 Loss: 0.5520126819610596
epocha # 17 Loss: 0.5488260388374329
epocha # 18 Loss: 0.5459555983543396
epocha # 19 Loss: 0.5433695912361145
epocha # 20 Loss: 0.5410398244857788
epocha # 21 Loss: 0.5389404892921448
epocha # 22 Loss: 0.5370488166809082
epocha # 23 Loss: 0.5353439450263977
epocha # 24 Loss: 0.5338073372840881
epocha # 25 Loss: 0.5324221849441528
epocha # 26 Loss: 0.5311734080314636
epocha # 27 Loss

In [8]:
X_new = X_test[:3]
y_pred = X_new @ w + b
print(y_pred)
print(y_test[:3])

tensor([[1.9179],
        [1.9562],
        [1.1253]], grad_fn=<AddBackward0>)
tensor([[1.0540],
        [3.4530],
        [1.0570]])


In [9]:
#torch hight API
import torch.nn as nn 

torch.manual_seed(52)
linear_model = nn.Linear(in_features=n_features, out_features=1)

print(linear_model.bias)
print(linear_model.weight)

Parameter containing:
tensor([-0.2398], requires_grad=True)
Parameter containing:
tensor([[ 0.1529,  0.2007,  0.1317, -0.0418,  0.3233,  0.1380, -0.1301,  0.3028]],
       requires_grad=True)


In [11]:
optimizer = torch.optim.SGD(params=linear_model.parameters(), lr=lr)
mse = torch.nn.MSELoss()

def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print("Epoch:", epoch, "Loss", loss.item())

In [25]:
train_bgd(linear_model, optimizer, mse, X_train, y_train, 30)

Epoch: 0 Loss 6.662117004394531
Epoch: 1 Loss 1.6106297969818115
Epoch: 2 Loss 0.8144829273223877
Epoch: 3 Loss 0.6768147945404053
Epoch: 4 Loss 0.6442325115203857
Epoch: 5 Loss 0.6293932199478149
Epoch: 6 Loss 0.6183295845985413
Epoch: 7 Loss 0.6087242364883423
Epoch: 8 Loss 0.6001229882240295
Epoch: 9 Loss 0.5923764109611511
Epoch: 10 Loss 0.5853918194770813
Epoch: 11 Loss 0.5790920853614807
Epoch: 12 Loss 0.5734090805053711
Epoch: 13 Loss 0.5682815909385681
Epoch: 14 Loss 0.5636544823646545
Epoch: 15 Loss 0.5594783425331116
Epoch: 16 Loss 0.5557084083557129
Epoch: 17 Loss 0.5523046255111694
Epoch: 18 Loss 0.5492308735847473
Epoch: 19 Loss 0.5464544892311096
Epoch: 20 Loss 0.5439460873603821
Epoch: 21 Loss 0.5416794419288635
Epoch: 22 Loss 0.5396307706832886
Epoch: 23 Loss 0.5377785563468933
Epoch: 24 Loss 0.5361036062240601
Epoch: 25 Loss 0.5345885157585144
Epoch: 26 Loss 0.5332176685333252
Epoch: 27 Loss 0.5319769382476807
Epoch: 28 Loss 0.5308536887168884
Epoch: 29 Loss 0.52983647

In [26]:
with torch.no_grad():
    print(linear_model(X_new))

tensor([[1.9013],
        [1.9492],
        [1.1506]])


### MLP

In [12]:
torch.manual_seed(52)

seq_model = torch.nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

optimizer = torch.optim.SGD(seq_model.parameters(), lr=0.1)
mse = torch.nn.MSELoss()

train_bgd(seq_model, optimizer, mse, X_train, y_train, 30)

Epoch: 0 Loss 6.2093377113342285
Epoch: 1 Loss 3.3030920028686523
Epoch: 2 Loss 1.8504443168640137
Epoch: 3 Loss 1.2772990465164185
Epoch: 4 Loss 1.1099869012832642
Epoch: 5 Loss 0.9839632511138916
Epoch: 6 Loss 0.886165976524353
Epoch: 7 Loss 0.813666045665741
Epoch: 8 Loss 0.7616143822669983
Epoch: 9 Loss 0.7248446345329285
Epoch: 10 Loss 0.6988593935966492
Epoch: 11 Loss 0.6801035404205322
Epoch: 12 Loss 0.6660177111625671
Epoch: 13 Loss 0.6549256443977356
Epoch: 14 Loss 0.6457038521766663
Epoch: 15 Loss 0.6376990675926208
Epoch: 16 Loss 0.63052898645401
Epoch: 17 Loss 0.6239607334136963
Epoch: 18 Loss 0.6178625822067261
Epoch: 19 Loss 0.6121400594711304
Epoch: 20 Loss 0.6067323088645935
Epoch: 21 Loss 0.6016130447387695
Epoch: 22 Loss 0.5967378616333008
Epoch: 23 Loss 0.5920771956443787
Epoch: 24 Loss 0.5876143574714661
Epoch: 25 Loss 0.5833315849304199
Epoch: 26 Loss 0.5792192220687866
Epoch: 27 Loss 0.5752744674682617
Epoch: 28 Loss 0.5714776515960693
Epoch: 29 Loss 0.56782454252

In [13]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
data_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)#, num_workers=4, persistent_workers=True)

In [14]:
torch.manual_seed(52)

seq_model2 = torch.nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

seq_model2 = seq_model2.to('cuda', non_blocking=True)

optimizer = torch.optim.SGD(params=seq_model2.parameters(), lr=0.02)
mse = torch.nn.MSELoss()

In [15]:
def train_minibatch(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to('cuda')
            y_batch = y_batch.to('cuda')
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch: {epoch}, loss: {mean_loss}")

In [16]:
train_minibatch(seq_model2, optimizer, mse, data_loader, 30)

Epoch: 0, loss: 0.6915812311608335
Epoch: 1, loss: 0.45175630373758496
Epoch: 2, loss: 0.4110041940472028
Epoch: 3, loss: 0.39077599178864364
Epoch: 4, loss: 0.376152739073116
Epoch: 5, loss: 0.3677501431136385
Epoch: 6, loss: 0.3572083205056826
Epoch: 7, loss: 0.3525730627645303
Epoch: 8, loss: 0.3400988822338368
Epoch: 9, loss: 0.3329762366327477
Epoch: 10, loss: 0.32873529663025325
Epoch: 11, loss: 0.32384271990083896
Epoch: 12, loss: 0.3235196027607086
Epoch: 13, loss: 0.32137224410117105
Epoch: 14, loss: 0.3171406637778005
Epoch: 15, loss: 0.3135769385077763
Epoch: 16, loss: 0.31102529155500863
Epoch: 17, loss: 0.30910996681408504
Epoch: 18, loss: 0.3106477608223227
Epoch: 19, loss: 0.3064108469476134
Epoch: 20, loss: 0.30765342680832086
Epoch: 21, loss: 0.3057132053389676
Epoch: 22, loss: 0.3026754968499733
Epoch: 23, loss: 0.299703107204189
Epoch: 24, loss: 0.3003766848807185
Epoch: 25, loss: 0.2986257140125547
Epoch: 26, loss: 0.3008650915818988
Epoch: 27, loss: 0.2971189333256

In [32]:
def evaluate_mb(model, data_loader, metrics_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            metrics.append(metrics_fn(y_pred, y_batch))            
            
    return aggregate_fn(torch.stack(metrics))

In [33]:
valid_dataset = TensorDataset(X_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32)

evaluate_mb(seq_model2, valid_loader, mse)

tensor(0.3316, device='cuda:0')

In [34]:
#implementing rmse instead of mse
evaluate_mb(seq_model2, valid_loader, mse, lambda metric: torch.sqrt(torch.mean(metric)))

tensor(0.5758, device='cuda:0')

In [35]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
            
    return metric.compute()

rmse = torchmetrics.MeanSquaredError(squared=False).to('cuda')

evaluate_tm(seq_model2, valid_loader, rmse)

tensor(0.5761, device='cuda:0')

In [36]:
class WideAndDeep(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_learning = nn.Sequential(nn.Linear(n_features, 50), nn.ReLU(),
                                           nn.Linear(50, 40), nn.ReLU())
        self.output_layer = nn.Linear(40 + n_features, 1)
        
    def forward(self, X):
        deep_proceed = self.deep_learning(X)
        X_full = torch.concat([X, deep_proceed], dim=1)
        return self.output_layer(X_full)

In [37]:
torch.manual_seed(52)
deep_model = WideAndDeep(n_features).to('cuda')

optimizer=torch.optim.SGD(deep_model.parameters(), lr=0.002)
mse = nn.MSELoss()

train_minibatch(deep_model, optimizer, mse, data_loader, 100)
#train_bgd(deep_model, optimizer, mse, )

Epoch: 0, loss: 1.4234062346650094
Epoch: 1, loss: 0.6499505731441784
Epoch: 2, loss: 0.5935073061226067
Epoch: 3, loss: 0.5934971498086435
Epoch: 4, loss: 0.5493345693222835
Epoch: 5, loss: 0.552323741464003
Epoch: 6, loss: 0.5395890768734652
Epoch: 7, loss: 0.5227110353783314
Epoch: 8, loss: 0.507878498397497
Epoch: 9, loss: 0.5160822006028155
Epoch: 10, loss: 0.500463476445138
Epoch: 11, loss: 0.4978411961121363
Epoch: 12, loss: 0.4889632489433011
Epoch: 13, loss: 0.4833765204535847
Epoch: 14, loss: 0.47906201872207926
Epoch: 15, loss: 0.4770034582469134
Epoch: 16, loss: 0.470661959315472
Epoch: 17, loss: 0.46822884000531123
Epoch: 18, loss: 0.46421686590583794
Epoch: 19, loss: 0.4604730081110832
Epoch: 20, loss: 0.4577074267024278
Epoch: 21, loss: 0.45404588259738526
Epoch: 22, loss: 0.4514362079431878
Epoch: 23, loss: 0.4474542190390695
Epoch: 24, loss: 0.44469528481111686
Epoch: 25, loss: 0.4428171800138298
Epoch: 26, loss: 0.4387361190437405
Epoch: 27, loss: 0.4366253180018926
E

In [38]:
evaluate_tm(deep_model, valid_loader, rmse)

tensor(0.5893, device='cuda:0')

In [39]:
class WideAndDeep2(nn.Module):
    def __init__(self):
        super().__init__()
        self.deep_learning = nn.Sequential(nn.Linear(6, 50), nn.ReLU(),
                                           nn.Linear(50, 40), nn.ReLU())
        self.output_layer = nn.Linear(5 + 40, 1)
        
    def forward(self, X):
        X_deep = X[:, 2:]
        X_out = X[:, :5]
        wide_learned = self.deep_learning(X_deep)
        X_full = torch.concat([X_out, wide_learned], dim=1)
        return self.output_layer(X_full)

In [40]:
deep_model2 = WideAndDeep2().to('cuda')

optimizer = torch.optim.SGD(deep_model2.parameters(), lr=0.002) 
mse = nn.MSELoss()

train_minibatch(deep_model2, optimizer, mse, data_loader, 100)

Epoch: 0, loss: 1.4023407515111328
Epoch: 1, loss: 0.6172698514657794
Epoch: 2, loss: 0.5599609385751927
Epoch: 3, loss: 0.5380177612238299
Epoch: 4, loss: 0.5197090308833642
Epoch: 5, loss: 0.5088623591470949
Epoch: 6, loss: 0.5035074348888443
Epoch: 7, loss: 0.4958964842067215
Epoch: 8, loss: 0.4900407538575641
Epoch: 9, loss: 0.485118166875031
Epoch: 10, loss: 0.47800578039870134
Epoch: 11, loss: 0.47386994871331184
Epoch: 12, loss: 0.4694772953704252
Epoch: 13, loss: 0.4656651566305692
Epoch: 14, loss: 0.4614812909979509
Epoch: 15, loss: 0.4584638041965032
Epoch: 16, loss: 0.4549888425459296
Epoch: 17, loss: 0.4527647429964444
Epoch: 18, loss: 0.4502597154990813
Epoch: 19, loss: 0.44773997297875817
Epoch: 20, loss: 0.44498355192364564
Epoch: 21, loss: 0.44290599847532647
Epoch: 22, loss: 0.4407895842418255
Epoch: 23, loss: 0.43834532645799346
Epoch: 24, loss: 0.436147071006223
Epoch: 25, loss: 0.434312879047007
Epoch: 26, loss: 0.432407872728805
Epoch: 27, loss: 0.4305676363101883


In [41]:
evaluate_tm(deep_model2, valid_loader, rmse)

tensor(0.6133, device='cuda:0')

### Multiple Inputs

In [56]:
class WideAndDeep3(nn.Module):
    def __init__(self):
        super().__init__()
        self.deep_learning = nn.Sequential(nn.Linear(6, 50), nn.ReLU(), nn.Linear(50, 40), nn.ReLU())
        self.output_layer = nn.Linear(5+40, 1)
        
    def forward(self, X_wide, X_deep):
        X_learned = self.deep_learning(X_deep)
        X_full = torch.concat([X_wide, X_learned], dim=1)
        return self.output_layer(X_full)

In [57]:
train_data_wd = TensorDataset(X_train[:, :5], X_train[:, 2:], y_train)
wd_train_loader = DataLoader(train_data_wd, batch_size=32, shuffle=True)

valid_data_wd = TensorDataset(X_valid[:, :5], X_valid[:, 2:], y_valid)
wd_valid_loader = DataLoader(valid_data_wd, batch_size=32, shuffle=True)

test_data_wd = TensorDataset(X_test[:, :5], X_test[:, 2:], y_test)
wd_test_loader = DataLoader(test_data_wd, batch_size=32, shuffle=True)

In [58]:
def train_wd3(model, optimizer, criterion, data_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for *X_batch, y_batch in data_loader:
            X_batch = [X.to('cuda') for X in X_batch]
            y_batch = y_batch.to('cuda')
            
            y_pred = model(*X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            
            optimizer.step()
            optimizer.zero_grad()
        
        mean_loss = total_loss / len(data_loader)
        print(f"Epoch# {epoch} Loss: {mean_loss}")

In [59]:
def evaluate_wd3(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch_wide, X_batch_deep, y_batch in data_loader:
            X_batch_wide, X_batch_deep, y_batch = X_batch_wide.to('cuda'), X_batch_deep.to('cuda'), y_batch.to('cuda')      
            y_pred = model(X_batch_wide, X_batch_deep)
            metric.update(y_pred, y_batch)
            
    return metric.compute()

In [60]:
wad3 = WideAndDeep3().to('cuda')

optimizer = torch.optim.SGD(wad3.parameters(), lr=0.01)
mse = nn.MSELoss()

train_wd3(wad3, optimizer, mse, wd_train_loader, 100)

Epoch# 0 Loss: 0.8216857190715199
Epoch# 1 Loss: 0.5639018634329697
Epoch# 2 Loss: 0.48374596532190683
Epoch# 3 Loss: 0.4606336505303371
Epoch# 4 Loss: 0.43978687608501815
Epoch# 5 Loss: 0.4330340508398652
Epoch# 6 Loss: 0.4225629762031553
Epoch# 7 Loss: 0.4109805114837882
Epoch# 8 Loss: 0.401913374012954
Epoch# 9 Loss: 0.3949131981873339
Epoch# 10 Loss: 0.3901357107207215
Epoch# 11 Loss: 0.38171571343415583
Epoch# 12 Loss: 0.376230307608915
Epoch# 13 Loss: 0.37110264579479113
Epoch# 14 Loss: 0.3649747631632098
Epoch# 15 Loss: 0.3596665344176223
Epoch# 16 Loss: 0.3600581437168918
Epoch# 17 Loss: 0.35446119631390305
Epoch# 18 Loss: 0.3514466358948562
Epoch# 19 Loss: 0.348368469941414
Epoch# 20 Loss: 0.3463636900120151
Epoch# 21 Loss: 0.3527445222361613
Epoch# 22 Loss: 0.34905756221628653
Epoch# 23 Loss: 0.3417493523285695
Epoch# 24 Loss: 0.3412156998209168
Epoch# 25 Loss: 0.3379778941195756
Epoch# 26 Loss: 0.3385393488024684
Epoch# 27 Loss: 0.34084403520274104
Epoch# 28 Loss: 0.33445857

In [61]:
evaluate_wd3(wad3, wd_valid_loader, rmse)

tensor(0.5855, device='cuda:0')

### Multiple Outputs

In [84]:
class WideAndDeep4(nn.Module):
    def __init__(self):
        super().__init__() 
        self.deep_learning = nn.Sequential(nn.Linear(6, 50), nn.ReLU(), nn.Linear(50, 40), nn.ReLU())
        self.output_layer = nn.Linear(5+40, 1)
        self.auxiliary_layer = nn.Linear(40, 1)
        
    def forward(self, X_wide, X_deep):
        X_learned = self.deep_learning(X_deep)
        X_full = torch.concat([X_learned, X_wide], dim=1)
        output_res = self.output_layer(X_full)
        auxiliary_res = self.auxiliary_layer(X_learned)
        return output_res, auxiliary_res

In [91]:
def train_wd4(model, optimizer, criterion, data_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_wide, X_deep, y_batch in data_loader:
            X_wide, X_deep, y_batch = X_wide.to('cuda'), X_deep.to('cuda'), y_batch.to('cuda')
            y_pred, y_pred_aux = model(X_wide, X_deep)
            main_loss = criterion(y_pred, y_batch)
            aux_loss = criterion(y_pred_aux, y_batch)
            loss = 0.8*main_loss + 0.2*aux_loss
            total_loss += loss.item()
            loss.backward()
            
            optimizer.step()
            optimizer.zero_grad()
        
        mean_loss = total_loss / len(data_loader)
        print(f"Epoch # {epoch}, Loss: {mean_loss}")

In [92]:
wad4 = WideAndDeep4().to('cuda')

optimizer = torch.optim.SGD(params=wad4.parameters(), lr=0.01)
mse = nn.MSELoss()

train_wd4(wad4, optimizer, mse, wd_train_loader, 100)

Epoch # 0, Loss: 0.9282555227995496
Epoch # 1, Loss: 0.6065520968188962
Epoch # 2, Loss: 0.5476472378137902
Epoch # 3, Loss: 0.5169288960193029
Epoch # 4, Loss: 0.4992485507132066
Epoch # 5, Loss: 0.4814451705240453
Epoch # 6, Loss: 0.4692660641944437
Epoch # 7, Loss: 0.45729634865721547
Epoch # 8, Loss: 0.44713737135504983
Epoch # 9, Loss: 0.435396130054684
Epoch # 10, Loss: 0.42970360317617007
Epoch # 11, Loss: 0.41593613917544736
Epoch # 12, Loss: 0.4044571535855748
Epoch # 13, Loss: 0.3983639171820576
Epoch # 14, Loss: 0.3920836485471333
Epoch # 15, Loss: 0.38699608359296445
Epoch # 16, Loss: 0.38623247178357106
Epoch # 17, Loss: 0.3758685793051131
Epoch # 18, Loss: 0.3714674213499769
Epoch # 19, Loss: 0.36915610384277225
Epoch # 20, Loss: 0.36623332003632125
Epoch # 21, Loss: 0.3678153525794389
Epoch # 22, Loss: 0.35819569962509606
Epoch # 23, Loss: 0.35602523050931695
Epoch # 24, Loss: 0.35702967934184154
Epoch # 25, Loss: 0.35319606671177445
Epoch # 26, Loss: 0.35391656879434863

In [93]:
def evaluate_wd4(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch_wide, X_batch_deep, y_batch in data_loader:
            X_batch_wide, X_batch_deep, y_batch = X_batch_wide.to('cuda'), X_batch_deep.to('cuda'), y_batch.to('cuda')      
            y_pred, _ = model(X_batch_wide, X_batch_deep)
            metric.update(y_pred, y_batch)
            
    return metric.compute()

In [94]:
evaluate_wd4(wad4, wd_valid_loader, rmse)

tensor(0.5837, device='cuda:0')

### Image classifier

In [95]:
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_valid_data = torchvision.datasets.FashionMNIST(root="datasets", train=True, download=True, transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(root="datasets", train=False, download=True, transform=toTensor)

torch.manual_seed(52)
train_data, valid_data = torch.utils.data.random_split(train_valid_data, [55000, 5000])

100.0%
100.0%
100.0%
100.0%


In [96]:
train_data_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_data_loader = DataLoader(valid_data, batch_size=32)
test_data_loader = DataLoader(test_data, batch_size=32)

In [104]:
X_instance, y_instance = train_data[1]
print(X_instance.shape)
train_valid_data.classes[y_instance]

torch.Size([1, 28, 28])


'Pullover'

In [111]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, hidden_1, hidden_2, n_outputs):
        super().__init__()
        self.mlp = nn.Sequential(nn.Flatten(), nn.Linear(n_inputs, hidden_1),
                                 nn.ReLU(), nn.Linear(hidden_1, hidden_2),
                                 nn.ReLU(), nn.Linear(hidden_2, n_outputs))
        
    def forward(self, X):
        return self.mlp(X)
    

torch.manual_seed(52)
img_cls = ImageClassifier(28*28, 300, 250, 10).to('cuda')
xentropy = nn.CrossEntropyLoss()

In [112]:
optimizer = torch.optim.SGD(params=img_cls.parameters(), lr=0.01)

train_minibatch(img_cls, optimizer, xentropy, train_data_loader, 20)

Epoch: 0, loss: 1.0715766273972043
Epoch: 1, loss: 0.5866352355258281
Epoch: 2, loss: 0.5041315067709012
Epoch: 3, loss: 0.46898114784399114
Epoch: 4, loss: 0.4456570092902896
Epoch: 5, loss: 0.42822384623644866
Epoch: 6, loss: 0.41246469416387416
Epoch: 7, loss: 0.3971287721660749
Epoch: 8, loss: 0.3857003766447392
Epoch: 9, loss: 0.3745618699689533
Epoch: 10, loss: 0.36432470929976324
Epoch: 11, loss: 0.35557825733333775
Epoch: 12, loss: 0.3469895010964686
Epoch: 13, loss: 0.33859832821724856
Epoch: 14, loss: 0.33207023187544266
Epoch: 15, loss: 0.3249290249223207
Epoch: 16, loss: 0.31885353618406015
Epoch: 17, loss: 0.31318619642720935
Epoch: 18, loss: 0.30785124031409683
Epoch: 19, loss: 0.30181006349757566


In [120]:
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to('cuda')

print(evaluate_tm(img_cls, train_data_loader, accuracy))
print(evaluate_tm(img_cls, valid_data_loader, accuracy))

tensor(0.8965, device='cuda:0')
tensor(0.8782, device='cuda:0')


In [122]:
img_cls.eval()
X_new, y_new = next(iter(valid_data_loader))
X_new = X_new[:3].to('cuda')

with torch.no_grad():
    y_pred_logits=img_cls(X_new)
    
y_pred = y_pred_logits.argmax(dim=1)
print([train_valid_data.classes[index] for index in y_pred])

['Pullover', 'T-shirt/top', 'Dress']

In [128]:
import torch.nn.functional as F

y_proba = F.softmax(y_pred_logits, dim=1)
y_proba.round(decimals=3)

tensor([[0.0040, 0.0010, 0.8790, 0.0030, 0.1020, 0.0000, 0.0080, 0.0000, 0.0020,
         0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0040, 0.0020, 0.0010, 0.9800, 0.0030, 0.0000, 0.0060, 0.0010, 0.0030,
         0.0000]], device='cuda:0')

In [130]:
y_top3_logits, y_top3_indexes = torch.topk(y_pred_logits, k=3, dim=1)
y_top3_logits = F.softmax(y_top3_logits, dim=1).round(decimals=3)

print(y_top3_logits)
print(y_top3_indexes) 

tensor([[0.8880, 0.1030, 0.0080],
        [1.0000, 0.0000, 0.0000],
        [0.9890, 0.0060, 0.0040]], device='cuda:0')
tensor([[2, 4, 6],
        [0, 6, 2],
        [3, 6, 0]], device='cuda:0')


### Fine-Tunning

In [138]:
import optuna

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 30, 300)
    
    model = ImageClassifier(n_inputs=28*28, hidden_1=n_hidden, hidden_2=n_hidden, n_outputs=10).to('cuda')
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    
    train_minibatch(model, optimizer, xentropy, train_data_loader, 10)
    
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to('cuda')
    validation_accuracy = evaluate_tm(model, valid_data_loader, accuracy)
    
    return validation_accuracy

In [139]:
torch.manual_seed(52)
sampler = optuna.samplers.TPESampler(seed=52)

study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=5)

[I 2026-02-10 17:13:08,332] A new study created in memory with name: no-name-12139f5c-fa5c-4dd0-8681-e28c4008fce4


Epoch: 0, loss: 0.900476960507571
Epoch: 1, loss: 0.5336202463658485
Epoch: 2, loss: 0.4761479727884862
Epoch: 3, loss: 0.4412468346513032
Epoch: 4, loss: 0.4160354860590737
Epoch: 5, loss: 0.396637641573937
Epoch: 6, loss: 0.37911552435665924
Epoch: 7, loss: 0.36890977499719413
Epoch: 8, loss: 0.35656004664317476
Epoch: 9, loss: 0.3478132116458248


[I 2026-02-10 17:14:25,013] Trial 0 finished with value: 0.8622000217437744 and parameters: {'learning_rate': 0.01960836411250069, 'n_hidden': 37}. Best is trial 0 with value: 0.8622000217437744.


Epoch: 0, loss: 2.3001402776694837
Epoch: 1, loss: 2.285361781239579
Epoch: 2, loss: 2.2707165481740477
Epoch: 3, loss: 2.25579218187604
Epoch: 4, loss: 2.240299409735404
Epoch: 5, loss: 2.223990585722155
Epoch: 6, loss: 2.206628372947999
Epoch: 7, loss: 2.187967137169741
Epoch: 8, loss: 2.1677720722866445
Epoch: 9, loss: 2.145773323834671


[I 2026-02-10 17:15:42,408] Trial 1 finished with value: 0.40400001406669617 and parameters: {'learning_rate': 6.967589559514088e-05, 'n_hidden': 197}. Best is trial 0 with value: 0.8622000217437744.


...

[I 2026-02-10 17:17:00,947] Trial 2 finished with value: 0.10840000212192535 and parameters: {'learning_rate': 2.472508887456302e-05, 'n_hidden': 198}. Best is trial 0 with value: 0.8622000217437744.


Epoch: 0, loss: 2.305847522614653
Epoch: 1, loss: 2.30286497969902
Epoch: 2, loss: 2.2999394681996006
Epoch: 3, loss: 2.297055389521911
Epoch: 4, loss: 2.2942126375634424
Epoch: 5, loss: 2.2914023435414013
Epoch: 6, loss: 2.288613126470711
Epoch: 7, loss: 2.2858565114416862
Epoch: 8, loss: 2.2831025158270215
Epoch: 9, loss: 2.280370834748921


[I 2026-02-10 17:18:17,879] Trial 3 finished with value: 0.10639999806880951 and parameters: {'learning_rate': 1.6427099148039897e-05, 'n_hidden': 290}. Best is trial 0 with value: 0.8622000217437744.


Epoch: 0, loss: 0.6216176163801764
Epoch: 1, loss: 0.41346142918668977
Epoch: 2, loss: 0.36798984651876926
Epoch: 3, loss: 0.3405276103732959
Epoch: 4, loss: 0.3196619957915186
Epoch: 5, loss: 0.3046761693529364
Epoch: 6, loss: 0.28935459710401873
Epoch: 7, loss: 0.27991998529819084
Epoch: 8, loss: 0.2689835580652054
Epoch: 9, loss: 0.2587521296057367


[I 2026-02-10 17:19:39,455] Trial 4 finished with value: 0.8826000094413757 and parameters: {'learning_rate': 0.08350596456454125, 'n_hidden': 171}. Best is trial 4 with value: 0.8826000094413757.


In [141]:
print(study.best_params)
print(study.best_value)

{'learning_rate': 0.08350596456454125, 'n_hidden': 171}
0.8826000094413757


In [142]:
img_cls_best = ImageClassifier(28*28, 171, 171, 10).to('cuda')
optimizer = torch.optim.SGD(params=img_cls_best.parameters(), lr=0.0835)
xentropy = nn.CrossEntropyLoss()

train_minibatch(img_cls_best, optimizer, xentropy, train_data_loader, n_epochs=15)

accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to('cuda')
print(evaluate_tm(img_cls_best, test_data_loader, accuracy))

Epoch: 0, loss: 0.6175169095509686
Epoch: 1, loss: 0.4115177854990945
Epoch: 2, loss: 0.36750367173731086
Epoch: 3, loss: 0.3392878882565285
Epoch: 4, loss: 0.3224630082993746
Epoch: 5, loss: 0.30422939598308946
Epoch: 6, loss: 0.29132415607372225
Epoch: 7, loss: 0.28049947710414136
Epoch: 8, loss: 0.2688303245659517
Epoch: 9, loss: 0.25980174025047165
Epoch: 10, loss: 0.2505312926174424
Epoch: 11, loss: 0.24364153466308236
Epoch: 12, loss: 0.2365604058111768
Epoch: 13, loss: 0.22994802620450447
Epoch: 14, loss: 0.2226256748992969
tensor(0.8865, device='cuda:0')


#### Saving model

In [143]:
torch.save(img_cls_best, 'img_cls_best.pt')

In [144]:
loaded_model = torch.load('img_cls_best.pt', weights_only=False)

In [145]:
torch.save(img_cls_best.state_dict(), 'img_cls_state_dict.pt')

In [146]:
new_model = ImageClassifier(28*28, 171, 171, 10).to('cuda')
loaded_weights = torch.load('img_cls_state_dict.pt', weights_only=True)

new_model.load_state_dict(loaded_weights)
new_model.eval()

ImageClassifier(
  (mlp): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=171, bias=True)
    (2): ReLU()
    (3): Linear(in_features=171, out_features=171, bias=True)
    (4): ReLU()
    (5): Linear(in_features=171, out_features=10, bias=True)
  )
)

In [147]:
model_data = {
    "model_state_dict" : img_cls_best.state_dict(),
    "model_hyperparameters" : {"n_inputs" : 28*28, "hidden_1" : 171, "hidden_2" : 171, "n_outputs" : 10}
}
torch.save(model_data, 'best_model_data.pt')

In [151]:
loaded_data = torch.load('best_model_data.pt', weights_only=True)
new_model = ImageClassifier(**loaded_data['model_hyperparameters'])
new_model.load_state_dict(loaded_data["model_state_dict"])
new_model.eval()

ImageClassifier(
  (mlp): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=171, bias=True)
    (2): ReLU()
    (3): Linear(in_features=171, out_features=171, bias=True)
    (4): ReLU()
    (5): Linear(in_features=171, out_features=10, bias=True)
  )
)

### optimizing

In [153]:
torchscript_model = torch.jit.trace(img_cls_best, X_new) #outdated
torchscript_model = torch.jit.script(img_cls_best)

optimized_model = torch.jit.optimize_for_inference(torchscript_model) #for inference

In [154]:
torchscript_model.save('torchscript_model.pt')

In [155]:
loaded_torchscript = torch.jit.load('torchscript_model.pt')

In [156]:
compiled_model = torch.compile(img_cls_best)